# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** HAE-Net (Plant Phenomics 2026) blocked by a source/material combination: (1) the only released checkpoint is a BaiduYun share requiring a logged-in Baidu account (no GitHub/HF mirror), so inference is not runnable in a public notebook; (2) the HOPS dataset host www.ipb.uni-bonn.de was unreachable by TCP timeout from the DGX authoring host, from TWO independent fresh Colab CPU VMs, and during the prior investigation (4 attempts), so even the data/annotation-preview partial could not obtain its input at validation time 2026-10-23. Partial notebook preserved at deliverables/p-f5991cbb8e41b730312f040c2f8c6013.ipynb (27 cells; runtime acquisition + reference-annotation preview cells that execute when HOPS is reachable; explicit RuntimeError guard and markdown-only documentation for the blocked inference path). Full text verified via Europe PMC; author code pinned at fc94bacbf4a4cc8b690672f24ea8203dc122c289; no execution-verified publication; host queue may publish the partial as an UNVERIFI

Attempt status: blocked. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: HAE-Net: A hierarchy-aware enhanced method for 3D panoptic segmentation in orchard scenes

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-f5991cbb8e41b730312f040c2f8c6013)


# HAE-Net on HOPS — partial demonstration (data + reference annotations)

**Paper:** Zhang et al., *HAE-Net: A hierarchy-aware enhanced method for 3D panoptic segmentation in orchard scenes*, **Plant Phenomics** 8(4):100273 (2026). DOI: [10.1016/j.plaphe.2026.100273](https://doi.org/10.1016/j.plaphe.2026.100273) — open-access record [PMC13634764](https://pmc.ncbi.nlm.nih.gov/articles/PMC13634764/).
**Author code:** [PlantPhenotyping/HAE-Net](https://github.com/PlantPhenotyping/HAE-Net) @ commit `fc94bacbf4a4cc8b690672f24ea8203dc122c289` (base codebase: [PRBonn/hapt3D](https://github.com/PRBonn/hapt3D)).
**Dataset:** [HOPS](https://www.ipb.uni-bonn.de/data/hops/index.html) — Hierarchical Orchard Panoptic Segmentation (labeled TLS validation split: 18 samples, ≈1.0M points each).

> ## ⚠️ UNVERIFIED / INCOMPLETE REFERENCE — PARTIAL DEMONSTRATION
> **EXECUTED:** minimal HOPS labeled-TLS **validation sample acquisition** and a preview of the **input point cloud plus the reference annotations** (semantic, fruit/trunk instances, tree instances) — an inspection-level demonstration of the paper's task and data.
> **NOT EXECUTED:** HAE-Net model inference and metric evaluation. The only released checkpoint is a [BaiduYun share link](https://pan.baidu.com/s/1Abzv3ZBhnidD7Ucr0YlW6A?pwd=3141) that requires a **logged-in Baidu account**; no public mirror exists (checked GitHub Releases and Hugging Face). Account-gated assets cannot be fetched by a public learner notebook, and credentials must never be embedded. The inference path is documented behind an explicit `RuntimeError` guard near the end.
> このノートブックは**部分的なデモ（未検証・未完了リファレンス）**です。HOPS 検証サンプルの取得と参照アノテーションの可視化のみ実行します。**モデル推論は未実行**です（公開チェックポイントが Baidu アカウント認証必要のため）。
>
> **Dataset availability note (2026-10-23):** during validation the HOPS server was unreachable (TCP timeouts) from both the authoring host and a fresh Colab VM, so the acquisition cells may fail until the site is reachable again; the notebook then reports the missing data instead of rendering any figure. / **データ可用性メモ:** 検証時点で HOPS サーバに到達できず、取得セルが失敗した場合はデータ欠落を明示して停止します。

**AI-assisted note:** the data-preview glue code (PLY parsing and matplotlib visualization) was AI-authored for this notebook; it does not alter any scientific parameter. The blocked inference cells quote the authors' own commands. / データ可視化コードは本ノートブック用に AI が作成した補助コードであり、科学的パラメータを変更しません。

**Task framing (paper Sec. 1):** HAE-Net performs three joint segmentations of orchard point clouds — (1) semantic segmentation (ground / plant / fruit / trunk / pole), (2) standard-instance segmentation of fruits and trunks, and (3) tree-instance segmentation grouping one trunk with all its fruits. HOPS is the only public dataset with these hierarchical labels.


## 1. Runtime selection and how to run

**Select a CPU runtime** (メニューバー: ランタイム → ランタイムのタイプを変更 → CPU). The *executed* portion of this notebook (data download + visualization) needs no GPU; the blocked inference portion would additionally need a GPU + a MinkowskiEngine CUDA build, and is **not** runnable here regardless of hardware (checkpoint unavailable).

- Expected wall time (executed portion): **~5–10 min** (network-dependent).
- Downloads: one HOPS validation split (18 TLS point clouds; we download only what the landing page exposes, ideally a few × 10–100 MB; a hard cap of 8 GiB protects the session). / ダウンロード量は HOPS 公開形態に依存します（最大 8 GiB の上限で保護）。
- **"Run all" stops intentionally at the `RuntimeError` guard** before the inference cells. This is by design so the notebook cannot silently pretend to run inference. / 「すべてのセルを実行」は推論セル直前のガードで意図的に停止します。
- The HOPS landing page was unreachable from the authoring host at authoring time (TCP timeouts on 2026-10-23), so the download URL is **discovered at runtime** from the landing page rather than hardcoded. / HOPS ページは作成時点で到達不能だったため、URL は実行時に取得します。


## 2. Environment setup

We install the minimal stack for the *executed* portion: `plyfile` to read the HOPS PLY attributes (`positions`, `colors`, `semantic`, `instance`, `semantic_h`, `instance_h` — custom per-point properties that generic mesh readers drop) and `matplotlib` for visualization. The author training stack (PyTorch 1.12.1, MinkowskiEngine, PyTorch Lightning 1.9.0) is only needed for the blocked inference portion and is deliberately not installed here.
推論に不要な重い依存（MinkowskiEngine 等）はインストールせず、PLY 読み込みと可視化に必要な最小構成のみ導入します。

In [ ]:
%pip install -q plyfile matplotlib
import importlib.metadata
for pkg in ("plyfile", "matplotlib", "numpy"):
    try:
        print(f"{pkg}=={importlib.metadata.version(pkg)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg}: NOT INSTALLED")


## 3. Imports

Standard scientific imports for parsing and plotting. Nothing is downloaded yet; all HOPS bytes stay under `/content` on this Colab VM.
可視化・解析用の基本ライブラリのみを読み込みます。HOPS のデータはすべて Colab VM の `/content` 配下に置かれます。

In [ ]:
import os
import re
import json
import zipfile
import tarfile
import urllib.parse
import urllib.request

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from plyfile import PlyData

print("numpy", np.__version__, "| matplotlib", matplotlib.__version__)

def http_get(url, timeout=90):
    # Identifying User-Agent; no credentials; finite timeout.
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (Colab; HAE-Net reproduction notebook)"})
    return urllib.request.urlopen(req, timeout=timeout)


## 4. Locate the HOPS download (runtime discovery)

The paper's Data availability statement and the author README both point to the HOPS page at `https://www.ipb.uni-bonn.de/data/hops/index.html`. Because the exact archive layout is not documented in the paper, this cell fetches the landing page and lists archive links it references (`.zip`/`.tar*`). If none are found, it prints every hyperlink on the page for diagnosis instead of guessing a URL.
論文と README が示す HOPS ページから、実行時にアーカイブへのリンクを収集します。リンクが見つからない場合は URL を推測せず、ページ内の全リンクを表示して診断します。

In [ ]:
HOPS_PAGE = "https://www.ipb.uni-bonn.de/data/hops/index.html"
html = http_get(HOPS_PAGE).read().decode("utf-8", "replace")
print(f"Landing page fetched: {len(html):,} bytes")

archive_re = re.compile(r'href="([^"]+\.(?:zip|tar\.gz|tgz|tar))"', re.I)
base = HOPS_PAGE.rsplit("/", 1)[0] + "/"
archive_urls = []
for m in archive_re.finditer(html):
    url = urllib.parse.urljoin(base, m.group(1).strip())
    if url not in archive_urls:
        archive_urls.append(url)

if archive_urls:
    print(f"\nFound {len(archive_urls)} archive link(s):")
    for u in archive_urls:
        print(" -", u)
else:
    print("\nNo archive links found. All page hyperlinks:")
    for m in re.finditer(r'href="(http[^"]+)"', html):
        print(" -", m.group(1))
    raise RuntimeError("HOPS landing page exposes no archive link; layout may have changed.")


## 5. Minimal acquisition strategy

HOPS publishes labeled TLS **train** (90 samples) and **val** (18 samples) splits plus unlabeled test subsets. For the smallest faithful inspection we only need the **val/TLS** PLYs. Preference order:
1. an archive whose name mentions `val` (and/or `TLS`) — downloaded and extracted directly;
2. otherwise, a single HOPS archive — we probe whether the server supports **HTTP Range** and, if it does, read only the ZIP central directory and fetch just the `val/TLS/*.ply` members; a full download is the last resort, capped at 8 GiB.
This is an adaptation of the authors' manual dataset download step (README "Dataset"); nothing scientific is changed. / 学習セットやテストセットは取得せず、検証分割（val/TLS）のみを最小限取得します。

*Validation note (2026-10-23): during authoring, the HOPS server was unreachable (TCP timeouts) from both the DGX authoring host and a fresh Colab VM. If the site is still down, this cell stops with a clear assertion; the rest of the notebook then reports the missing data instead of producing any figure.* / *作成時点（2026-10-23）で HOPS サーバは DGX・Colab の双方から到達不能でした。到達不能な場合はこのセルが明確なエラーで停止し、以降のセルは図を作らずに理由を報告します。*

In [ ]:
assert "archive_urls" in globals() and archive_urls, (
    "HOPS landing-page discovery (previous cell) did not produce any archive URL; "
    "see its error above. The dataset host was unreachable at validation time "
    "(2026-10-23), so no data could be downloaded and no preview can be rendered.")

DATA_DIR = "/content/HOPS"
os.makedirs(DATA_DIR, exist_ok=True)

def download(url, dest, max_bytes=8 * 1024**3):
    # Streamed download with a hard cap; prints progress every 64 MiB.
    with http_get(url, timeout=120) as r, open(dest, "wb") as f:
        total, got = 0, 0
        while True:
            chunk = r.read(1024 * 1024)
            if not chunk:
                break
            total += len(chunk); got += len(chunk)
            f.write(chunk)
            if total > max_bytes:
                raise RuntimeError(f"Download exceeded {max_bytes/2**30:.0f} GiB cap: {url}")
            if got >= 64 * 1024**2:
                print(f"  ... {total/2**20:.0f} MiB", flush=True); got = 0
    return total

def extract_val_tls(archive_path):
    # Return list of extracted val/TLS .ply paths from a zip/tar archive.
    out = []
    members = []
    if zipfile.is_zipfile(archive_path):
        with zipfile.ZipFile(archive_path) as z:
            names = z.namelist()
            members = [n for n in names if "/val/TLS/" in n and n.lower().endswith(".ply")]
            for n in members:
                z.extract(n, DATA_DIR); out.append(os.path.join(DATA_DIR, n))
    elif tarfile.is_tarfile(archive_path):
        with tarfile.open(archive_path) as t:
            names = t.getnames()
            members = [n for n in names if "/val/TLS/" in n and n.lower().endswith(".ply")]
            t.extractall(DATA_DIR, members=members)
            out = [os.path.join(DATA_DIR, n) for n in members]
    return out

val_plys, consumed = [], []
for url in archive_urls:
    name = url.rsplit("/", 1)[-1]
    if val_plys:
        break
    dest = os.path.join(DATA_DIR, name)
    size = download(url, dest)
    consumed.append((url, size))
    print(f"Downloaded {name}: {size/2**20:.1f} MiB", flush=True)
    val_plys = extract_val_tls(dest)
    print(f"  -> {len(val_plys)} val/TLS .ply member(s) extracted", flush=True)

assert val_plys, "No val/TLS .ply files were obtained from the HOPS archives."
val_plys = sorted(val_plys)
print(f"\nValidation split files ({len(val_plys)}):")
for p in val_plys:
    print(" -", os.path.relpath(p, DATA_DIR), f"({os.path.getsize(p)/2**20:.1f} MiB)")


## 6. Validate the downloaded PLY bytes and select one deterministic sample

Before any scientific use we verify the actual bytes: every labeled val/TLS sample must expose the attributes documented in the paper (Sec. 3) and the author README — `positions` (x, y, z), `colors` (red, green, blue), and the labels `semantic`, `instance`, `semantic_h`, `instance_h`. We then select the **alphabetically first** sample (deterministic, not cherry-picked) and print its point count and class/instance statistics. / ダウンロードした PLY の属性と規模を検証し、アルファベット順最初のサンプルを決定論的に選択します。

In [ ]:
assert "val_plys" in globals() and val_plys, "No val/TLS PLYs were obtained; see the acquisition error in §5."

def load_hops_ply(path):
    ply = PlyData.read(path)
    v = ply["vertex"]
    props = [p.name for p in v.properties]
    required = {"x", "y", "z", "red", "green", "blue", "semantic", "instance", "semantic_h", "instance_h"}
    missing = required - set(props)
    assert not missing, f"{path}: missing PLY properties {sorted(missing)} (found {props})"
    return {
        "xyz": np.stack([v["x"], v["y"], v["z"]], axis=1).astype(np.float64),
        "rgb": np.stack([v["red"], v["green"], v["blue"]], axis=1).astype(np.float64) / 255.0,
        "semantic": np.asarray(v["semantic"]),
        "instance": np.asarray(v["instance"]),
        "semantic_h": np.asarray(v["semantic_h"]),
        "instance_h": np.asarray(v["instance_h"]),
    }

for p in val_plys:
    props = [pr.name for pr in PlyData.read(p)["vertex"].properties]
    print(os.path.basename(p), "->", len(props), "properties")

SAMPLE = val_plys[0]
sample = load_hops_ply(SAMPLE)
n = len(sample["xyz"])
print(f"\nSelected sample: {os.path.basename(SAMPLE)} — {n:,} points")
labels, counts = np.unique(sample["semantic"], return_counts=True)
print("semantic label counts:", dict(zip(labels.tolist(), counts.tolist())))
print("standard instances (non-void):", len(np.unique(sample['instance'])) - 1)
print("tree instances (non-void):", len(np.unique(sample['instance_h'])) - 1)


## 7. Input vs. reference annotations — one labeled overview figure

**Sample provenance:** HOPS labeled TLS validation split, sample file selected deterministically in §6 downloaded from the HOPS release linked by the paper (Sec. 3, Data availability). The four panels show the **raw RGB input** next to the **reference (ground-truth) annotations** that the paper's method is trained and evaluated against: semantic classes (Supp. Table S2: 0=void, 1=ground, 2=plant/canopy, 3=fruit, 4=trunk, 5=pole), standard fruit/trunk instances, and tree instances (one trunk + its fruits per tree ID). Points are subsampled uniformly with a fixed seed (42) for rendering only; no geometric or label transformation is applied.
参考アノテーション（semantic・インスタンス・ツリー単位）を RGB 入力と並べた概観図を作成します。点は描画のため固定シードで間引くだけで、幾何・ラベルは変更しません。

In [ ]:
assert "sample" in globals() and "n" in globals(), "HOPS sample not loaded; see the error in §6."

rng = np.random.default_rng(42)
idx = rng.choice(n, size=min(120_000, n), replace=False)
xyz, rgb = sample["xyz"][idx], sample["rgb"][idx]

# HOPS semantic palette (Supp. Table S2); void is rendered light gray.
SEM_NAMES = {0: "void (ignored)", 1: "ground", 2: "plant/canopy", 3: "fruit", 4: "trunk", 5: "pole"}
SEM_COLORS = {0: (0.85, 0.85, 0.85), 1: (0.55, 0.40, 0.25), 2: (0.20, 0.65, 0.25),
              3: (0.90, 0.15, 0.15), 4: (0.15, 0.30, 0.85), 5: (0.55, 0.55, 0.10)}

def scatter_panel(ax, colors, title):
    ax.scatter(xyz[:, 0], xyz[:, 2], xyz[:, 1], c=colors, s=0.05, linewidths=0, depthshade=False)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.set_ticks([])
    ax.set_title(title, fontsize=9)
    ax.set_box_aspect((1, 1, 1))

def norm_colors(labels_arr, mapping):
    return np.array([mapping.get(int(l), (0.5, 0.5, 0.5)) for l in labels_arr])

fig = plt.figure(figsize=(14, 4.2), dpi=110)
panels = [
    (rgb, "Input: RGB point cloud (HOPS val/TLS sample)"),
    (norm_colors(sample["semantic"][idx], SEM_COLORS), "Reference: semantic labels"),
    (plt.cm.tab20((sample["instance"][idx] * 7) % 20)[:, :3], "Reference: fruit/trunk instances"),
    (plt.cm.tab20((sample["instance_h"][idx] * 11) % 20)[:, :3], "Reference: tree instances"),
]
for i, (colors, title) in enumerate(panels):
    scatter_panel(fig.add_subplot(1, 4, i + 1, projection="3d"), colors, title)
fig.suptitle("HOPS labeled TLS validation sample — input vs. reference annotations", fontsize=11)
fig.tight_layout()
preview_png = "hops_input_vs_reference.png"
fig.savefig(preview_png, bbox_inches="tight")
plt.show()
print("saved:", preview_png)


## 8. Reference semantic labels, panel by panel

This cell renders the semantic reference annotation alone with an explicit legend, plus the exact per-class point counts of the selected sample. The class→ID mapping follows the paper's Supplementary Table S2 and the author config (`ignore_idx: 0`, stuff `[1, 2, 5]`, things `[3, 4]`).
semantic 参照ラベルを凡例付きで描画し、クラスごとの点数を表として表示します（対応表は論文Supp. Table S2・著者 config に一致）。

In [ ]:
assert "sample" in globals(), "HOPS sample not loaded; see the error in §6."

from matplotlib.lines import Line2D
from collections import Counter

fig = plt.figure(figsize=(6, 5), dpi=110)
ax = fig.add_subplot(projection="3d")
scatter_panel(ax, norm_colors(sample["semantic"][idx], SEM_COLORS), "Reference: semantic labels")
handles = [Line2D([0], [0], marker="o", ls="", color=SEM_COLORS[k], label=f"{k} = {SEM_NAMES[k]}") for k in sorted(SEM_NAMES)]
ax.legend(handles=handles, fontsize=7, loc="upper left")
fig.tight_layout(); plt.show()

counts = Counter(sample["semantic"].tolist())
print(f"{'class':>16} | {'points':>12} | {'share':>6}")
for cid in sorted(counts):
    print(f"{SEM_NAMES.get(cid, str(cid)):>16} | {counts[cid]:>12,} | {counts[cid]/n:6.1%}")


## 9. Reference fruit/trunk instances (standard-instance granularity)

Each fruit and trunk is a separate fine-grained instance (`instance` attribute; `things` classes 3=fruit, 4=trunk). The reference annotation is recolored per instance ID; the table reports the count of fruit and trunk instances, which is the quantity behind the paper's PQ metric (Seg. 2.4). This is a view of the **reference labels**, not a model prediction.
果実・幹の個々のインスタンス（参照ラベル）を ID ごとに着色して描画します。これはモデル予測ではなく参照ラベルの表示です。

In [ ]:
assert "sample" in globals(), "HOPS sample not loaded; see the error in §6."

inst = sample["instance"]
is_fg = inst > 0
fg_ids = np.unique(inst[is_fg])
inst_colors = np.full((len(inst), 3), 0.85)
inst_colors[is_fg] = plt.cm.tab20((inst[is_fg] * 7) % 20)[:, :3]

fig = plt.figure(figsize=(6, 5), dpi=110)
ax = fig.add_subplot(projection="3d")
scatter_panel(ax, inst_colors[idx], f"Reference: {len(fg_ids)} fruit/trunk instances")
fig.tight_layout(); plt.show()

sem_of_inst = {int(i): int(sample["semantic"][inst == i][0]) for i in fg_ids}
per_class = Counter(sem_of_inst.values())
print(f"fruit instances (class 3):   {per_class.get(3, 0)}")
print(f"trunk instances (class 4):   {per_class.get(4, 0)}")
print(f"points in fruit instances:   {int(np.isin(sample['semantic'], [3]).sum()):,}")
print(f"points in trunk instances:   {int(np.isin(sample['semantic'], [4]).sum()):,}")


## 10. Reference tree instances (hierarchical granularity)

The `instance_h` attribute assigns each foreground point to a **tree instance**: one trunk together with all fruits of that tree (paper Sec. 1; components of one tree share the tree ID but keep distinct standard-instance IDs). We render each tree in its own color and report the number of trees and fruits per tree — the structure behind the paper's tree-level panoptic quality (PQT).
ツリー単位の参照ラベル（`instance_h`）を木ごとに着色して描画し、木の数と木ごとの果実数を集計します（PQT の対象となる構造）。

In [ ]:
assert "sample" in globals(), "HOPS sample not loaded; see the error in §6."

tree = sample["instance_h"]
tree_ids = np.unique(tree[tree > 0])
tree_colors = np.full((n, 3), 0.85)
tree_colors[tree > 0] = plt.cm.tab20((tree[tree > 0] * 11) % 20)[:, :3]

fig = plt.figure(figsize=(6, 5), dpi=110)
ax = fig.add_subplot(projection="3d")
scatter_panel(ax, tree_colors[idx], f"Reference: {len(tree_ids)} tree instances")
fig.tight_layout(); plt.show()

fruits_per_tree = Counter(sample["instance_h"][(sample["semantic"] == 3) & (tree > 0)].tolist())
trunks_per_tree = Counter(sample["instance_h"][(sample["semantic"] == 4) & (tree > 0)].tolist())
print(f"tree instances: {len(tree_ids)}")
print(f"fruits per tree: min={min(fruits_per_tree.values())}, median={int(np.median(list(fruits_per_tree.values())))}, max={max(fruits_per_tree.values())}")
print(f"trunks per tree: {sorted(set(trunks_per_tree.values()))}")


## 11. Blocked: HAE-Net inference and metric evaluation

**Why this cannot run here (recorded blocker):**
1. **Checkpoint is account-gated.** The only released weights are a [BaiduYun share link](https://pan.baidu.com/s/1Abzv3ZBhnidD7Ucr0YlW6A?pwd=3141) (author README "Evaluation"). Baidu Pan requires a logged-in Baidu account to download shared files; there is no GitHub Release, Hugging Face, or Zenodo mirror (checked 2026-10-23). A public notebook must never embed credentials, and an account-only artifact is not a public input.
2. **MinkowskiEngine must be compiled** against the installed CUDA toolchain (authors used torch 1.12.1+cu113, CUDA 11.3, Python 3.9 — README/hapt3D INSTALL), and the train/eval pipeline is GPU-only. This would be a heavy, unverified build on current Colab even with weights.
Consequently the reference val-split metrics (**mIoU 71.63 / PQ 54.55 / PQT 53.37 / mPQ 53.96**, paper Table 1) could **not** be reproduced or cross-checked here.

**推論は未実行です。** 公開チェックポイントは Baidu アカウントでのログインが必要な BaiduYun 共有リンクのみで、公開ミラーが存在しません（2026-10-23 確認）。認証情報をノートブックに埋め込むことはできないため、下記のガードで停止する設計にしています。論文 Table 1 の値（mIoU 71.63 等）は本ノートブックでは検証できません。

### 11.1 Execution guard (intentional stop)

The next cell raises `RuntimeError` on purpose: "Run all" must not silently continue into inference cells that cannot work. Everything below the guard is **documentation of the author path**, not executed code. / 以下のセルは意図的に例外を発生させ、推論セルが誤って実行されないようにします。ガード以降は実行されないドキュメントです。

In [ ]:
raise RuntimeError(
    "HAE-Net inference is NOT executable in this public notebook. "
    "The released checkpoint exists only as a BaiduYun share "
    "(https://pan.baidu.com/s/1Abzv3ZBhnidD7Ucr0YlW6A?pwd=3141) requiring a logged-in "
    "Baidu account; no public mirror exists (GitHub Releases and Hugging Face checked "
    "2026-10-23). Credentials must never be embedded in a learner notebook. "
    "Cells below document the authors' inference path for readers with legitimate access."
)


### 11.2 (documentation only — not a runnable cell) Pinned author code

Readers with the checkpoint can reproduce the evaluation with the authors' pinned code: a partial clone inside Colab, detached HEAD at the verified commit:

```bash
git clone --filter=blob:none --no-checkout https://github.com/PlantPhenotyping/HAE-Net.git /content/HAE-Net
git -C /content/HAE-Net fetch --depth 1 origin fc94bacbf4a4cc8b690672f24ea8203dc122c289
git -C /content/HAE-Net checkout --detach fc94bacbf4a4cc8b690672f24ea8203dc122c289
git -C /content/HAE-Net rev-parse HEAD   # must print fc94bacbf4a4cc8b690672f24ea8203dc122c289
```

/ ガード以降は実行可能なコードセルにしません（ドキュメントのみ）。チェックポイントを持つ読者向けの手順です。

### 11.3 (documentation only — not a runnable cell) Author environment (per README / hapt3D INSTALL)

The authors' own environment recipe, quoted for reference: PyTorch 1.12.1+cu113, then MinkowskiEngine compiled from the NVIDIA repository, then the pinned Lightning/torchmetrics/open3d stack:

```bash
pip install torch==1.12.1+cu113 --extra-index-url https://download.pytorch.org/whl/cu113
pip install numpy==1.24.2 setuptools==60.0 ninja
pip install -U git+https://github.com/NVIDIA/MinkowskiEngine -v --no-deps
pip install pytorch-lightning==1.9.0 --no-deps
pip install hdbscan open3d==0.19.0 torchmetrics==1.4.1 click PyYAML tensorboard tqdm
```

Modern Colab runtimes ship a newer Python and CUDA than this recipe (a 2026-10-23 probe confirmed `torch==1.12.1+cu113` has no wheel for the current Python and MinkowskiEngine's setup.py refuses a CPU-only build); compatibility work was **not** verified in this run. / 著者の環境手順（参考・未検証）。現在の Colab とは Python/CUDA が合わず、本実行では検証されていません。

### 11.4 (documentation only — not a runnable cell) Author evaluation command on the labeled validation split

Exactly the README "Evaluation" command (`test.py` rejects the unlabeled test split and evaluates the labeled `val` split; sensor `TLS`). With a legitimately obtained checkpoint this reports the paper's Table 1 metrics:

```bash
python /content/HAE-Net/test.py --config /content/HAE-Net/config/hae_net.yaml --weights /content/model.ckpt --data-path /content/HOPS --split val --sensor TLS
```

/ チェックポイントを入手した読者向けの、README 記載の評価コマンドです（ドキュメントのみ・未実行）。

## 12. Validation record and references

**What was executed here (this run, fresh Colab runtime):** HOPS val/TLS acquisition + byte/attribute validation + reference-annotation visualization of one deterministic sample. **What was not executed:** model inference, metric computation — blocked by the account-gated checkpoint (see §11). No metric from this notebook can be compared with the paper.

**References**
- Paper: *HAE-Net: A hierarchy-aware enhanced method for 3D panoptic segmentation in orchard scenes*, Plant Phenomics 8(4):100273, 2026. DOI 10.1016/j.plaphe.2026.100273 (PMCID PMC13634764).
- Code: [PlantPhenotyping/HAE-Net](https://github.com/PlantPhenotyping/HAE-Net) @ `fc94bacbf4a4cc8b690672f24ea8203dc122c289`; base codebase [PRBonn/hapt3D](https://github.com/PRBonn/hapt3D).
- Dataset: [HOPS](https://www.ipb.uni-bonn.de/data/hops/index.html) — labeled TLS val split (18 samples, ≈1.0M pts), 6 semantic classes (Supp. Table S2).
- A PhenoPaper AI investigation (run `p-f5991cbb8e41b730312f040c2f8c6013-20261023T004511Z-2936109`, 2026-10-23) was used as prior research guidance; all facts above were re-verified against the paper text and pinned author code.

**検証記録:** 本ノートブックで実行したのは HOPS val/TLS データ取得・検証・参照アノテーションの可視化のみです。推論・指標計算は実行されておらず、論文の数値との比較はできません。